# ClarifySQL · Notebook 03 — Paraphrasing the training requests

**Goal.** The rule-based drafts of the 180 vague and unanswerable *training* requests are sometimes clumsy
("grade high point"). The base model rewrites each one from the original question plus a plain description
of the detail to hide. Every rewrite must pass the same automatic checks as the drafts (hidden value gone,
gold SQL still runs, nothing leaked...); otherwise the draft is kept. Validation and test requests are
**not** touched: the team edits those, so the tuned model is never tested on its own writing style.

**Time:** about 10 minutes (most of it loading the model). **GPU quota:** about 10 minutes.
**Settings:** Kaggle → *Session options* → **GPU T4 x2**, **Internet On**. Colab → *T4 GPU*.

In [ ]:
!nvidia-smi

## 1. Get the project code

This cell clones the project from GitHub (branch `main`) into the notebook's working folder and makes
the `clarifysql` package importable. Running it again later just pulls the newest commits.

*Private repository?* Add a secret called `GITHUB_TOKEN` (Kaggle: **Add-ons → Secrets**; Colab: the
**key icon** in the left bar) and set `USE_TOKEN = True`. Our repository is public, so this is not needed.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Sathviktr/LMA_ClarifySQL.git"
BRANCH = "main"
USE_TOKEN = False  # only needed for a private repository

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/LMA_ClarifySQL"

if USE_TOKEN:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

sys.path.insert(0, f"{REPO}/src")
print("Running on:", "Kaggle" if ON_KAGGLE else "Colab / other", "| repo at", REPO)

In [ ]:
# Libraries that are not preinstalled (or too old) on Kaggle/Colab. PyTorch with CUDA is already there.
!pip install -q -r {REPO}/requirements-gpu.txt
!pip install -q -e {REPO} --no-deps  # lets the scripts import clarifysql

## 2. Download Spider (the checks run the gold SQL on the real databases)

In [ ]:
os.environ["CLARIFYSQL_DATA"] = "/tmp/clarifysql_data"
!python {REPO}/scripts/download_data.py --only spider

## 3. Check the pipeline without the model (seconds)

A fake "model" that returns the rule drafts unchanged: almost every draft should be accepted (they passed
the same checks when the episodes were built).

In [ ]:
!python {REPO}/scripts/paraphrase_train.py --backend mock
!rm -f {REPO}/results/episodes/paraphrases_train_mock.jsonl

## 4. Paraphrase with the base model (FP16, greedy, batches of 16)

The output shows how many rewrites were accepted and the most common reasons for rejection.

In [ ]:
!python {REPO}/scripts/paraphrase_train.py --batch-size 16

## 5. A few examples: draft → rewrite

In [ ]:
import json
rows = [json.loads(l) for l in open(f"{REPO}/results/episodes/paraphrases_train.jsonl", encoding="utf-8")]
for r in rows[:12]:
    mark = "OK " if r["accepted"] else "NO "
    print(f"{mark} draft: {r['draft']}\n    model: {r['paraphrase']}\n    {'; '.join(r['reasons'])}\n")

## 6. Pack the result

In [ ]:
!cd {REPO} && zip -q {WORK}/checkpoint3_results.zip results/episodes/paraphrases_train.jsonl
!ls -la {WORK}/checkpoint3_results.zip

## 7. Send the result back

Download `checkpoint3_results.zip` (Kaggle: **Output** panel; Colab: **folder icon**) and unzip it
**inside the repository folder**, so that `results/episodes/paraphrases_train.jsonl` lands in place. Then
**stop the session**.